#Instalação e Download do Arquivo

In [2]:
!pip install kaggle


In [3]:
from google.colab import files
files.upload()   # vai pedir para você selecionar o kaggle.json


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"lucasborges12","key":"d2b0dc051b844c8388f25d8890f63d29"}'}

In [4]:
!mkdir -p /root/.kaggle /root/.config/kaggle \
 && cp kaggle.json /root/.kaggle/ \
 && cp kaggle.json /root/.config/kaggle/ \
 && chmod 600 /root/.kaggle/kaggle.json \
 && chmod 600 /root/.config/kaggle/kaggle.json


In [5]:
from kaggle.api.kaggle_api_extended import KaggleApi
import zipfile

# Inicializa a API
api = KaggleApi()
api.authenticate()

# Baixa o zip da competição
api.competition_download_files('nfl-big-data-bowl-2026-prediction', path='.')

# Extrai o conteúdo
with zipfile.ZipFile("nfl-big-data-bowl-2026-prediction.zip", 'r') as zip_ref:
    zip_ref.extractall("nfl_big_data_2026")

print("✅ Download e extração concluídos!")


✅ Download e extração concluídos!


In [8]:
import pandas as pd
import os
import glob

# Exemplo: carregando um arquivo de treino e o sample_submission
test_input_file = "/content/nfl_big_data_2026/test_input.csv"
sample_sub = "/content/nfl_big_data_2026/sample_submission.csv"
test_file = "/content/nfl_big_data_2026/test.csv"

df_train = pd.read_csv(test_input_file)
df_sample = pd.read_csv(sample_sub)
df_test = pd.read_csv(test_file)

# Diretório de treino
train_dir = "/content/nfl_big_data_2026/train/"

# Lista todos os arquivos de input e output
input_files = sorted(glob.glob(os.path.join(train_dir, "input_*.csv")))
output_files = sorted(glob.glob(os.path.join(train_dir, "output_*.csv")))

# Verifica se a quantidade de arquivos bate
assert len(input_files) == len(output_files), "Número de arquivos de input e output não coincide!"

# Listas para armazenar DataFrames
train_inputs = []
train_outputs = []

# Lê todos os arquivos e concatena
for inp_file, out_file in zip(input_files, output_files):
    df_inp = pd.read_csv(inp_file)
    df_out = pd.read_csv(out_file)

    train_inputs.append(df_inp)
    train_outputs.append(df_out)

# Concatena todos os DataFrames em um único
df_train_input = pd.concat(train_inputs, ignore_index=True)
df_train_output = pd.concat(train_outputs, ignore_index=True)

print("Treino Inputs:", df_train_input.shape)
print("Treino Outputs:", df_train_output.shape)

# Exemplo: mostrar primeiras linhas
df_train_input.head(), df_train_output.head()

print("Treino:", df_train.shape)
print("Sample submission:", df_sample.shape)
print("Teste:", df_test.shape)


Treino Inputs: (4880579, 23)
Treino Outputs: (562936, 6)
Treino: (49753, 23)
Sample submission: (5837, 3)
Teste: (5837, 4)


In [7]:
def show_type_tags(df):
    # Colunas do DataFrame
    colunas = df.columns

    # Tipos de cada coluna
    tipos_colunas = df.dtypes

    # Rótulos de classificação (tipo object)
    print("Rótulos de Classificação (object):")
    classificacao = tipos_colunas[tipos_colunas == 'object']
    display(classificacao)
    classificacao = classificacao.index.tolist()

    # Rótulos de regressão ou numéricos
    print("\nRótulos de Regressão ou Numéricos:")
    regressao = tipos_colunas[tipos_colunas != 'object']
    display(regressao)

    regressao = regressao.index.tolist()

    return classificacao, regressao



# Analise Exploratória dos Dados

## Analise Exploratoria dos Dados - Test.csv

In [9]:
# Agora funciona para qualquer DataFrame
test_regression, test_classification = show_type_tags(df_test)

for tag in test_regression:
  print(f"{tag}\n{df_train_input[tag].unique()}")


Rótulos de Classificação (object):


,0



Rótulos de Regressão ou Numéricos:


,0
game_id,int64
play_id,int64
nfl_id,int64
frame_id,int64


In [10]:
for tag in test_classification:
  print(f"{tag}\n{df_train_input[tag].unique()}")

game_id
[2023090700 2023091000 2023091001 2023091002 2023091003 2023091004
 2023091005 2023091006 2023091007 2023091008 2023091009 2023091010
 2023091011 2023091012 2023091013 2023091100 2023091400 2023091700
 2023091701 2023091702 2023091703 2023091704 2023091705 2023091706
 2023091707 2023091708 2023091709 2023091710 2023091711 2023091712
 2023091800 2023091801 2023092100 2023092400 2023092401 2023092402
 2023092403 2023092404 2023092405 2023092406 2023092407 2023092408
 2023092409 2023092410 2023092411 2023092412 2023092500 2023092501
 2023092800 2023100100 2023100101 2023100102 2023100103 2023100104
 2023100105 2023100106 2023100107 2023100108 2023100109 2023100110
 2023100111 2023100112 2023100113 2023100200 2023100500 2023100800
 2023100801 2023100802 2023100803 2023100804 2023100805 2023100806
 2023100807 2023100808 2023100809 2023100810 2023100811 2023100900
 2023101200 2023101500 2023101501 2023101502 2023101503 2023101504
 2023101505 2023101506 2023101507 2023101508 202310150

## EDA - Test_input.csv

In [11]:
test_input_regression, test_input_classification = show_type_tags(df_train_input)

Rótulos de Classificação (object):


,0
play_direction,object
player_name,object
player_height,object
player_birth_date,object
player_position,object
player_side,object
player_role,object



Rótulos de Regressão ou Numéricos:


,0
game_id,int64
play_id,int64
player_to_predict,bool
nfl_id,int64
frame_id,int64
absolute_yardline_number,int64
player_weight,int64
x,float64
y,float64
s,float64


In [12]:
for tag in test_input_regression:
  print(f"{tag}\n{df_train_input[tag].unique()}")


play_direction
['right' 'left']
player_name
['Bryan Cook' 'Justin Reid' "L'Jarius Sneed" ... 'Brady Russell'
 'Blake Gillikin' 'Matt Prater']
player_height
['6-1' '6-0' '5-11' '6-4' '6-3' '5-10' '5-9' '5-8' '6-2' '6-5' '6-6' '5-7'
 '6-7' '6-8' '5-6' '6-9']
player_birth_date
['1999-09-07' '1997-02-15' '1997-01-21' ... '1992-12-30' '1998-08-25'
 '1984-08-10']
player_position
['FS' 'SS' 'CB' 'MLB' 'WR' 'TE' 'QB' 'OLB' 'ILB' 'RB' 'DE' 'FB' 'NT' 'DT'
 'S' 'T' 'LB' 'P' 'K']
player_side
['Defense' 'Offense']
player_role
['Defensive Coverage' 'Other Route Runner' 'Passer' 'Targeted Receiver']


In [13]:
for tag in test_input_classification:
  print(f"{tag}\n{df_train_input[tag].unique()}")

game_id
[2023090700 2023091000 2023091001 2023091002 2023091003 2023091004
 2023091005 2023091006 2023091007 2023091008 2023091009 2023091010
 2023091011 2023091012 2023091013 2023091100 2023091400 2023091700
 2023091701 2023091702 2023091703 2023091704 2023091705 2023091706
 2023091707 2023091708 2023091709 2023091710 2023091711 2023091712
 2023091800 2023091801 2023092100 2023092400 2023092401 2023092402
 2023092403 2023092404 2023092405 2023092406 2023092407 2023092408
 2023092409 2023092410 2023092411 2023092412 2023092500 2023092501
 2023092800 2023100100 2023100101 2023100102 2023100103 2023100104
 2023100105 2023100106 2023100107 2023100108 2023100109 2023100110
 2023100111 2023100112 2023100113 2023100200 2023100500 2023100800
 2023100801 2023100802 2023100803 2023100804 2023100805 2023100806
 2023100807 2023100808 2023100809 2023100810 2023100811 2023100900
 2023101200 2023101500 2023101501 2023101502 2023101503 2023101504
 2023101505 2023101506 2023101507 2023101508 202310150

##Analise Exploratória dos Dados - Train_Input

In [14]:
train_input_regressao, train_input_classificacao = show_type_tags(df_train_input)


Rótulos de Classificação (object):


,0
play_direction,object
player_name,object
player_height,object
player_birth_date,object
player_position,object
player_side,object
player_role,object



Rótulos de Regressão ou Numéricos:


,0
game_id,int64
play_id,int64
player_to_predict,bool
nfl_id,int64
frame_id,int64
absolute_yardline_number,int64
player_weight,int64
x,float64
y,float64
s,float64


In [15]:
for tag in train_input_regressao:
  print(f"{tag}\n{df_train_input[tag].unique()}")


play_direction
['right' 'left']
player_name
['Bryan Cook' 'Justin Reid' "L'Jarius Sneed" ... 'Brady Russell'
 'Blake Gillikin' 'Matt Prater']
player_height
['6-1' '6-0' '5-11' '6-4' '6-3' '5-10' '5-9' '5-8' '6-2' '6-5' '6-6' '5-7'
 '6-7' '6-8' '5-6' '6-9']
player_birth_date
['1999-09-07' '1997-02-15' '1997-01-21' ... '1992-12-30' '1998-08-25'
 '1984-08-10']
player_position
['FS' 'SS' 'CB' 'MLB' 'WR' 'TE' 'QB' 'OLB' 'ILB' 'RB' 'DE' 'FB' 'NT' 'DT'
 'S' 'T' 'LB' 'P' 'K']
player_side
['Defense' 'Offense']
player_role
['Defensive Coverage' 'Other Route Runner' 'Passer' 'Targeted Receiver']


In [16]:
for tag in train_input_classificacao:
  print(f"{tag}\n{df_train_input[tag].unique()}")

game_id
[2023090700 2023091000 2023091001 2023091002 2023091003 2023091004
 2023091005 2023091006 2023091007 2023091008 2023091009 2023091010
 2023091011 2023091012 2023091013 2023091100 2023091400 2023091700
 2023091701 2023091702 2023091703 2023091704 2023091705 2023091706
 2023091707 2023091708 2023091709 2023091710 2023091711 2023091712
 2023091800 2023091801 2023092100 2023092400 2023092401 2023092402
 2023092403 2023092404 2023092405 2023092406 2023092407 2023092408
 2023092409 2023092410 2023092411 2023092412 2023092500 2023092501
 2023092800 2023100100 2023100101 2023100102 2023100103 2023100104
 2023100105 2023100106 2023100107 2023100108 2023100109 2023100110
 2023100111 2023100112 2023100113 2023100200 2023100500 2023100800
 2023100801 2023100802 2023100803 2023100804 2023100805 2023100806
 2023100807 2023100808 2023100809 2023100810 2023100811 2023100900
 2023101200 2023101500 2023101501 2023101502 2023101503 2023101504
 2023101505 2023101506 2023101507 2023101508 202310150

##Analise Exploratória dos Dados - Train_Output

In [17]:
train_output_regressao, train_output_classificacao = show_type_tags(df_train_output)

Rótulos de Classificação (object):


,0



Rótulos de Regressão ou Numéricos:


,0
game_id,int64
play_id,int64
nfl_id,int64
frame_id,int64
x,float64
y,float64


In [18]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error


class ParticipantVisibleError(Exception):
    pass


TARGET = ['x', 'y']


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Compute RMSE for NFL competition.
    Expected input:
      - solution and submission as pandas.DataFrame
      - Column 'id': unique identifier for each (game_id, play_id, nfl_id, frame_id)
      - Column 'x'
      - Column 'y'
    Examples
    --------
    >>> import pandas as pd
    >>> row_id_column_name = 'id'
    >>> solution = pd.DataFrame({'id': ['21_12_2_1', '21_12_2_2', '21_12_2_3'], 'x': [1,2,3], 'y':[4,2,3]})
    >>> submission  = pd.DataFrame({'id': ['21_12_2_1', '21_12_2_2', '21_12_2_3'], 'x': [1.1,2,3], 'y':[4,2.2,3]})
    >>> round(score(solution, submission, row_id_column_name=row_id_column_name), 4)
    0.0913
    >>> submission  = pd.DataFrame({'id': ['21_12_2_1', '21_12_2_2', '21_12_2_3'], 'x': [0,2,3], 'y':[4,2.2,3]})
    >>> round(score(solution, submission, row_id_column_name=row_id_column_name), 4)
    0.4163
    >>> submission  = pd.DataFrame({'id': ['21_12_2_1', '21_12_2_2', '21_12_2_3'], 'x': [1,2,1], 'y':[4,0,3]})
    >>> round(score(solution, submission, row_id_column_name=row_id_column_name), 4)
    1.1547
    """

    if row_id_column_name not in solution.columns:
        raise ParticipantVisibleError(f"Solution file missing required column: '{row_id_column_name}'")
    if row_id_column_name not in submission.columns:
        raise ParticipantVisibleError(f"Submission file missing required column: '{row_id_column_name}'")

    missing_in_solution = set(TARGET) - set(solution.columns)
    missing_in_submission = set(TARGET) - set(submission.columns)

    if missing_in_solution:
        raise ParticipantVisibleError(f'Solution file missing required columns: {missing_in_solution}')
    if missing_in_submission:
        raise ParticipantVisibleError(f'Submission file missing required columns: {missing_in_submission}')

    if solution.shape[0] != submission.shape[0]:
        raise ParticipantVisibleError(
            f'Prediction rows mismatch!, expected: {solution.shape[0]}, found: {submission.shape[0]}'
        )

    if not np.all(solution[row_id_column_name] == submission[row_id_column_name]):
        raise ParticipantVisibleError('Submission has a different order or different row ids than the solution')

    return mean_squared_error(solution[TARGET], submission[TARGET], squared=False)

# Treinamento da IA

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import sklearn
import sklearn.datasets
import sklearn.linear_model
import scipy.io

def sigmoid(x):
    """
    Compute the sigmoid of x

    Arguments:
    x -- A scalar or numpy array of any size.

    Return:
    s -- sigmoid(x)
    """
    s = 1/(1+np.exp(-x))
    return s

def relu(x):
    """
    Compute the relu of x

    Arguments:
    x -- A scalar or numpy array of any size.

    Return:
    s -- relu(x)
    """
    s = np.maximum(0,x)

    return s

def load_planar_dataset(seed):

    np.random.seed(seed)

    m = 400 # number of examples
    N = int(m/2) # number of points per class
    D = 2 # dimensionality
    X = np.zeros((m,D)) # data matrix where each row is a single example
    Y = np.zeros((m,1), dtype='uint8') # labels vector (0 for red, 1 for blue)
    a = 4 # maximum ray of the flower

    for j in range(2):
        ix = range(N*j,N*(j+1))
        t = np.linspace(j*3.12,(j+1)*3.12,N) + np.random.randn(N)*0.2 # theta
        r = a*np.sin(4*t) + np.random.randn(N)*0.2 # radius
        X[ix] = np.c_[r*np.sin(t), r*np.cos(t)]
        Y[ix] = j

    X = X.T
    Y = Y.T

    return X, Y

def initialize_parameters(layer_dims):
    """
    Arguments:
    layer_dims -- python array (list) containing the dimensions of each layer in our network

    Returns:
    parameters -- python dictionary containing your parameters "W1", "b1", ..., "WL", "bL":
                    W1 -- weight matrix of shape (layer_dims[l], layer_dims[l-1])
                    b1 -- bias vector of shape (layer_dims[l], 1)
                    Wl -- weight matrix of shape (layer_dims[l-1], layer_dims[l])
                    bl -- bias vector of shape (1, layer_dims[l])

    Tips:
    - For example: the layer_dims for the "Planar Data classification model" would have been [2,2,1].
    This means W1's shape was (2,2), b1 was (1,2), W2 was (2,1) and b2 was (1,1). Now you have to generalize it!
    - In the for loop, use parameters['W' + str(l)] to access Wl, where l is the iterative integer.
    """

    np.random.seed(3)
    parameters = {}
    L = len(layer_dims) # number of layers in the network

    for l in range(1, L):
        parameters['W' + str(l)] = np.random.randn(layer_dims[l], layer_dims[l-1]) / np.sqrt(layer_dims[l-1])
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))

        assert(parameters['W' + str(l)].shape == layer_dims[l], layer_dims[l-1])
        assert(parameters['W' + str(l)].shape == layer_dims[l], 1)


    return parameters

def forward_propagation(X, parameters):
    """
    Implements the forward propagation (and computes the loss) presented in Figure 2.

    Arguments:
    X -- input dataset, of shape (input size, number of examples)
    parameters -- python dictionary containing your parameters "W1", "b1", "W2", "b2", "W3", "b3":
                    W1 -- weight matrix of shape ()
                    b1 -- bias vector of shape ()
                    W2 -- weight matrix of shape ()
                    b2 -- bias vector of shape ()
                    W3 -- weight matrix of shape ()
                    b3 -- bias vector of shape ()

    Returns:
    loss -- the loss function (vanilla logistic loss)
    """

    # retrieve parameters
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    W3 = parameters["W3"]
    b3 = parameters["b3"]

    # LINEAR -> RELU -> LINEAR -> RELU -> LINEAR -> SIGMOID
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)
    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3)

    cache = (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3)

    return A3, cache

def backward_propagation(X, Y, cache):
    """
    Implement the backward propagation presented in figure 2.

    Arguments:
    X -- input dataset, of shape (input size, number of examples)
    Y -- true "label" vector (containing 0 if cat, 1 if non-cat)
    cache -- cache output from forward_propagation()

    Returns:
    gradients -- A dictionary with the gradients with respect to each parameter, activation and pre-activation variables
    """
    m = X.shape[1]
    (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3) = cache

    dZ3 = A3 - Y
    dW3 = 1./m * np.dot(dZ3, A2.T)
    db3 = 1./m * np.sum(dZ3, axis=1, keepdims = True)

    dA2 = np.dot(W3.T, dZ3)
    dZ2 = np.multiply(dA2, np.int64(A2 > 0))
    dW2 = 1./m * np.dot(dZ2, A1.T)
    db2 = 1./m * np.sum(dZ2, axis=1, keepdims = True)

    dA1 = np.dot(W2.T, dZ2)
    dZ1 = np.multiply(dA1, np.int64(A1 > 0))
    dW1 = 1./m * np.dot(dZ1, X.T)
    db1 = 1./m * np.sum(dZ1, axis=1, keepdims = True)

    gradients = {"dZ3": dZ3, "dW3": dW3, "db3": db3,
                 "dA2": dA2, "dZ2": dZ2, "dW2": dW2, "db2": db2,
                 "dA1": dA1, "dZ1": dZ1, "dW1": dW1, "db1": db1}

    return gradients

def update_parameters(parameters, grads, learning_rate):
    """
    Update parameters using gradient descent

    Arguments:
    parameters -- python dictionary containing your parameters:
                    parameters['W' + str(i)] = Wi
                    parameters['b' + str(i)] = bi
    grads -- python dictionary containing your gradients for each parameters:
                    grads['dW' + str(i)] = dWi
                    grads['db' + str(i)] = dbi
    learning_rate -- the learning rate, scalar.

    Returns:
    parameters -- python dictionary containing your updated parameters
    """

    n = len(parameters) // 2 # number of layers in the neural networks

    # Update rule for each parameter
    for k in range(n):
        parameters["W" + str(k+1)] = parameters["W" + str(k+1)] - learning_rate * grads["dW" + str(k+1)]
        parameters["b" + str(k+1)] = parameters["b" + str(k+1)] - learning_rate * grads["db" + str(k+1)]

    return parameters

def predict(X, y, parameters):
    """
    This function is used to predict the results of a  n-layer neural network.

    Arguments:
    X -- data set of examples you would like to label
    parameters -- parameters of the trained model

    Returns:
    p -- predictions for the given dataset X
    """

    m = X.shape[1]
    p = np.zeros((1,m), dtype = int)

    # Forward propagation
    a3, caches = forward_propagation(X, parameters)

    # convert probas to 0/1 predictions
    for i in range(0, a3.shape[1]):
        if a3[0,i] > 0.5:
            p[0,i] = 1
        else:
            p[0,i] = 0

    # print results

    #print ("predictions: " + str(p[0,:]))
    #print ("true labels: " + str(y[0,:]))
    print("Accuracy: "  + str(np.mean((p[0,:] == y[0,:]))))

    return p

def compute_cost(a3, Y):
    """
    Implement the cost function

    Arguments:
    a3 -- post-activation, output of forward propagation
    Y -- "true" labels vector, same shape as a3

    Returns:
    cost - value of the cost function
    """
    m = Y.shape[1]

    logprobs = np.multiply(-np.log(a3),Y) + np.multiply(-np.log(1 - a3), 1 - Y)
    cost = 1./m * np.nansum(logprobs)

    return cost

def load_dataset():
    train_dataset = h5py.File('datasets/train_catvnoncat.h5', "r")
    train_set_x_orig = np.array(train_dataset["train_set_x"][:]) # your train set features
    train_set_y_orig = np.array(train_dataset["train_set_y"][:]) # your train set labels

    test_dataset = h5py.File('datasets/test_catvnoncat.h5', "r")
    test_set_x_orig = np.array(test_dataset["test_set_x"][:]) # your test set features
    test_set_y_orig = np.array(test_dataset["test_set_y"][:]) # your test set labels

    classes = np.array(test_dataset["list_classes"][:]) # the list of classes

    train_set_y = train_set_y_orig.reshape((1, train_set_y_orig.shape[0]))
    test_set_y = test_set_y_orig.reshape((1, test_set_y_orig.shape[0]))

    train_set_x_orig = train_set_x_orig.reshape(train_set_x_orig.shape[0], -1).T
    test_set_x_orig = test_set_x_orig.reshape(test_set_x_orig.shape[0], -1).T

    train_set_x = train_set_x_orig/255
    test_set_x = test_set_x_orig/255

    return train_set_x, train_set_y, test_set_x, test_set_y, classes


def predict_dec(parameters, X):
    """
    Used for plotting decision boundary.

    Arguments:
    parameters -- python dictionary containing your parameters
    X -- input data of size (m, K)

    Returns
    predictions -- vector of predictions of our model (red: 0 / blue: 1)
    """

    # Predict using forward propagation and a classification threshold of 0.5
    a3, cache = forward_propagation(X, parameters)
    predictions = (a3>0.5)
    return predictions

def load_planar_dataset(randomness, seed):

    np.random.seed(seed)

    m = 50
    N = int(m/2) # number of points per class
    D = 2 # dimensionality
    X = np.zeros((m,D)) # data matrix where each row is a single example
    Y = np.zeros((m,1), dtype='uint8') # labels vector (0 for red, 1 for blue)
    a = 2 # maximum ray of the flower

    for j in range(2):

        ix = range(N*j,N*(j+1))
        if j == 0:
            t = np.linspace(j, 4*3.1415*(j+1),N) #+ np.random.randn(N)*randomness # theta
            r = 0.3*np.square(t) + np.random.randn(N)*randomness # radius
        if j == 1:
            t = np.linspace(j, 2*3.1415*(j+1),N) #+ np.random.randn(N)*randomness # theta
            r = 0.2*np.square(t) + np.random.randn(N)*randomness # radius

        X[ix] = np.c_[r*np.cos(t), r*np.sin(t)]
        Y[ix] = j

    X = X.T
    Y = Y.T

    return X, Y

def plot_decision_boundary(model, X, y):
    # Set min and max values and give it some padding
    x_min, x_max = X[0, :].min() - 1, X[0, :].max() + 1
    y_min, y_max = X[1, :].min() - 1, X[1, :].max() + 1
    h = 0.01
    # Generate a grid of points with distance h between them
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    # Predict the function value for the whole grid
    Z = model(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    # Plot the contour and training examples
    plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral)
    plt.ylabel('x2')
    plt.xlabel('x1')
    plt.scatter(X[0, :], X[1, :], c=y, cmap=plt.cm.Spectral)
    plt.show()

def load_2D_dataset():
    data = scipy.io.loadmat(data_file_path)
    train_X = data['X'].T
    train_Y = data['y'].T
    test_X = data['Xval'].T
    test_Y = data['yval'].T

    plt.scatter(train_X[0, :], train_X[1, :], c=train_Y, s=40, cmap=plt.cm.Spectral);

    return train_X, train_Y, test_X, test_Y

<>:85: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:86: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:85: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:86: SyntaxWarning: assertion is always true, perhaps remove parentheses?
/tmp/ipython-input-607237232.py:85: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(parameters['W' + str(l)].shape == layer_dims[l], layer_dims[l-1])
/tmp/ipython-input-607237232.py:86: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(parameters['W' + str(l)].shape == layer_dims[l], 1)


In [2]:
import numpy as np

def compute_cost_with_regularization_test_case():
    np.random.seed(1)
    Y_assess = np.array([[1, 1, 0, 1, 0]])
    W1 = np.random.randn(2, 3)
    b1 = np.random.randn(2, 1)
    W2 = np.random.randn(3, 2)
    b2 = np.random.randn(3, 1)
    W3 = np.random.randn(1, 3)
    b3 = np.random.randn(1, 1)
    parameters = {"W1": W1, "b1": b1, "W2": W2, "b2": b2, "W3": W3, "b3": b3}
    a3 = np.array([[ 0.40682402,  0.01629284,  0.16722898,  0.10118111,  0.40682402]])
    return a3, Y_assess, parameters

def backward_propagation_with_regularization_test_case():
    np.random.seed(1)
    X_assess = np.random.randn(3, 5)
    Y_assess = np.array([[1, 1, 0, 1, 0]])
    cache = (np.array([[-1.52855314,  3.32524635,  2.13994541,  2.60700654, -0.75942115],
         [-1.98043538,  4.1600994 ,  0.79051021,  1.46493512, -0.45506242]]),
  np.array([[ 0.        ,  3.32524635,  2.13994541,  2.60700654,  0.        ],
         [ 0.        ,  4.1600994 ,  0.79051021,  1.46493512,  0.        ]]),
  np.array([[-1.09989127, -0.17242821, -0.87785842],
         [ 0.04221375,  0.58281521, -1.10061918]]),
  np.array([[ 1.14472371],
         [ 0.90159072]]),
  np.array([[ 0.53035547,  5.94892323,  2.31780174,  3.16005701,  0.53035547],
         [-0.69166075, -3.47645987, -2.25194702, -2.65416996, -0.69166075],
         [-0.39675353, -4.62285846, -2.61101729, -3.22874921, -0.39675353]]),
  np.array([[ 0.53035547,  5.94892323,  2.31780174,  3.16005701,  0.53035547],
         [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
         [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ]]),
  np.array([[ 0.50249434,  0.90085595],
         [-0.68372786, -0.12289023],
         [-0.93576943, -0.26788808]]),
  np.array([[ 0.53035547],
         [-0.69166075],
         [-0.39675353]]),
  np.array([[-0.3771104 , -4.10060224, -1.60539468, -2.18416951, -0.3771104 ]]),
  np.array([[ 0.40682402,  0.01629284,  0.16722898,  0.10118111,  0.40682402]]),
  np.array([[-0.6871727 , -0.84520564, -0.67124613]]),
  np.array([[-0.0126646]]))
    return X_assess, Y_assess, cache

def forward_propagation_with_dropout_test_case():
    np.random.seed(1)
    X_assess = np.random.randn(3, 5)
    W1 = np.random.randn(2, 3)
    b1 = np.random.randn(2, 1)
    W2 = np.random.randn(3, 2)
    b2 = np.random.randn(3, 1)
    W3 = np.random.randn(1, 3)
    b3 = np.random.randn(1, 1)
    parameters = {"W1": W1, "b1": b1, "W2": W2, "b2": b2, "W3": W3, "b3": b3}

    return X_assess, parameters

def backward_propagation_with_dropout_test_case():
    np.random.seed(1)
    X_assess = np.random.randn(3, 5)
    Y_assess = np.array([[1, 1, 0, 1, 0]])
    cache = (np.array([[-1.52855314,  3.32524635,  2.13994541,  2.60700654, -0.75942115],
           [-1.98043538,  4.1600994 ,  0.79051021,  1.46493512, -0.45506242]]), np.array([[ True, False,  True,  True,  True],
           [ True,  True,  True,  True, False]], dtype=bool), np.array([[ 0.        ,  0.        ,  4.27989081,  5.21401307,  0.        ],
           [ 0.        ,  8.32019881,  1.58102041,  2.92987024,  0.        ]]), np.array([[-1.09989127, -0.17242821, -0.87785842],
           [ 0.04221375,  0.58281521, -1.10061918]]), np.array([[ 1.14472371],
           [ 0.90159072]]), np.array([[ 0.53035547,  8.02565606,  4.10524802,  5.78975856,  0.53035547],
           [-0.69166075, -1.71413186, -3.81223329, -4.61667916, -0.69166075],
           [-0.39675353, -2.62563561, -4.82528105, -6.0607449 , -0.39675353]]), np.array([[ True, False,  True, False,  True],
           [False,  True, False,  True,  True],
           [False, False,  True, False, False]], dtype=bool), np.array([[ 1.06071093,  0.        ,  8.21049603,  0.        ,  1.06071093],
           [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
           [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ]]), np.array([[ 0.50249434,  0.90085595],
           [-0.68372786, -0.12289023],
           [-0.93576943, -0.26788808]]), np.array([[ 0.53035547],
           [-0.69166075],
           [-0.39675353]]), np.array([[-0.7415562 , -0.0126646 , -5.65469333, -0.0126646 , -0.7415562 ]]), np.array([[ 0.32266394,  0.49683389,  0.00348883,  0.49683389,  0.32266394]]), np.array([[-0.6871727 , -0.84520564, -0.67124613]]), np.array([[-0.0126646]]))


    return X_assess, Y_assess, cache


In [3]:
def model(X, Y, learning_rate = 0.3, num_iterations = 30000, print_cost = True, lambd = 0, keep_prob = 1):
    """
    Implements a three-layer neural network: LINEAR->RELU->LINEAR->RELU->LINEAR->SIGMOID.

    Arguments:
    X -- input data, of shape (input size, number of examples)
    Y -- true "label" vector (1 for blue dot / 0 for red dot), of shape (output size, number of examples)
    learning_rate -- learning rate of the optimization
    num_iterations -- number of iterations of the optimization loop
    print_cost -- If True, print the cost every 10000 iterations
    lambd -- regularization hyperparameter, scalar
    keep_prob - probability of keeping a neuron active during drop-out, scalar.

    Returns:
    parameters -- parameters learned by the model. They can then be used to predict.
    """

    grads = {}
    costs = []                            # to keep track of the cost
    m = X.shape[1]                        # number of examples
    layers_dims = [X.shape[0], 20, 3, 1]

    # Initialize parameters dictionary.
    parameters = initialize_parameters(layers_dims)

    # Loop (gradient descent)

    for i in range(0, num_iterations):

        # Forward propagation: LINEAR -> RELU -> LINEAR -> RELU -> LINEAR -> SIGMOID.
        if keep_prob == 1:
            a3, cache = forward_propagation(X, parameters)
        elif keep_prob < 1:
            a3, cache = forward_propagation_with_dropout(X, parameters, keep_prob)

        # Cost function
        if lambd == 0:
            cost = compute_cost(a3, Y)
        else:
            cost = compute_cost_with_regularization(a3, Y, parameters, lambd)

        # Backward propagation.
        assert(lambd==0 or keep_prob==1)    # it is possible to use both L2 regularization and dropout,
                                            # but this assignment will only explore one at a time
        if lambd == 0 and keep_prob == 1:
            grads = backward_propagation(X, Y, cache)
        elif lambd != 0:
            grads = backward_propagation_with_regularization(X, Y, cache, lambd)
        elif keep_prob < 1:
            grads = backward_propagation_with_dropout(X, Y, cache, keep_prob)

        # Update parameters.
        parameters = update_parameters(parameters, grads, learning_rate)

        # Print the loss every 10000 iterations
        if print_cost and i % 10000 == 0:
            print("Cost after iteration {}: {}".format(i, cost))
        if print_cost and i % 1000 == 0:
            costs.append(cost)

    # plot the cost
    plt.plot(costs)
    plt.ylabel('cost')
    plt.xlabel('iterations (x1,000)')
    plt.title("Learning rate =" + str(learning_rate))
    plt.show()

    return parameters


In [4]:
parameters = model(train_X, train_Y)
print ("On the training set:")
predictions_train = predict(train_X, train_Y, parameters)
print ("On the test set:")
predictions_test = predict(test_X, test_Y, parameters)

NameError: name 'train_X' is not defined